# Tree Processing
For this study, trees were modeled as if they were 'buildings' but with a high wall absorption coefficient. 

# Glasgow Tree Processing
Data:  https://data.ubdc.ac.uk/dataset/96ad6074-acff-49fc-95da-473fe6da4205 

## Reading all the files into 1 gdf 

In [ ]:
import geopandas as gpd
import pandas as pd
import glob
import os

# Folder containing the shapefiles
folder_path = r"C:\Users\ibk1\NoiseModelling\Glasgow\env_data\tree_volume\tree_canopy_products\treetop_location"

# Get a list of all shapefiles in the folder
shapefiles = glob.glob(os.path.join(folder_path, "*.shp"))

# Read and process each shapefile
gdfs = [gpd.read_file(shp).set_crs(epsg=27700, allow_override=True) for shp in shapefiles]

# Merge all shapefiles into a single GeoDataFrame
merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

# Create a 0.5-meter buffer around each point
merged_gdf["geometry"] = merged_gdf.geometry.buffer(1) # This sets the average diameter to 1 (0.5m, 50cm radius), might be too large 

print(f"Merged and buffered gdf created!")

In [ ]:
len(merged_gdf)

In [ ]:
merged_gdf.head()

In [ ]:
merged_gdf["HEIGHT"].median()

In [ ]:
merged_gdf.columns
merged_gdf = merged_gdf[['Z', "geometry"]]
merged_gdf = merged_gdf[['Z', 'geometry']].rename(columns={'Z': 'HEIGHT', 'geometry': 'THE_GEOM'})
merged_gdf["G"] = .1 # Wall absorption 

In [ ]:
merged_gdf = merged_gdf.set_geometry("THE_GEOM")

In [ ]:
merged_gdf.head()

In [ ]:
merged_gdf.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\noisemodelling_outputs\summerbuildings\trees.geojson")

## Combining with Buildings 

In [ ]:
import geopandas as gpd
buildings = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\noisemodelling_outputs\Round2\ready_input_buildings_cleaned.geojson") # Changed to a different srid 
buildings.columns

In [ ]:
len(buildings)

In [ ]:
test = buildings[buildings["HEIGHT"] < 0]

In [ ]:
test.explore()

In [ ]:
buildings.columns

In [ ]:
buildings.head()

In [ ]:
# Changing geometry names to match case with noisemodelling 
#buildings = buildings[['Ground_Z', "geometry"]]
#buildings = buildings[['Ground_Z', 'geometry']].rename(columns={'Ground_Z': 'HEIGHT', 'geometry': 'geometry'})

In [ ]:
buildings["G"] = .9 # Wall absorption 

In [ ]:
buildings.head()

In [ ]:
trees_clean = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\noisemodelling_outputs\summerbuildings\trees_clean.geojson")

In [ ]:
trees_clean.head()

In [ ]:
trees_clean["G"] = .1 # Wall absorption 

In [ ]:
combined_gdf = gpd.GeoDataFrame(pd.concat([trees_clean, buildings], ignore_index=True))

In [ ]:
combined_gdf.head()

In [ ]:
combined_gdf["G"].unique()

In [ ]:
subset = gpd.clip(combined_gdf, bbox)

In [ ]:
subset.explore()

In [ ]:
len(combined_gdf)

In [ ]:
combined_gdf.columns

In [ ]:
combined_gdf = combined_gdf.drop(columns=['PK'])

In [ ]:
combined_gdf = combined_gdf[['HEIGHT', 'geometry', 'G']].rename(columns={'HEIGHT': 'HEIGHT', 'geometry': 'THE_GEOM', 'G': 'G'})

In [ ]:
combined_gdf.columns

In [ ]:
# Changing geometry names to match case with noisemodelling 
combined_gdf = combined_gdf.set_geometry("THE_GEOM")
# Save as a geojson, note that this is now a 27700
combined_gdf.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\noisemodelling_outputs\Round3\trees_final_4.geojson", driver="GeoJSON")

# Edinburgh Trees and Buildings
Data: https://data.edinburghcouncilmaps.info/datasets/6c224ada230b4d349cfd739015c7846d_41/explore 

#### buildings layer 
Acquired from OpenStreetMap

In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
nm_edi_buildings = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\noisemodelling_inputs\buildings\nm_edi_buildings.geojson")

#### setting buildings wall abs to 0.9

In [ ]:
nm_edi_buildings["G"] = 0.9 # Adding wall absorption 

In [ ]:
nm_edi_buildings.head()

#### reading trees shapefiles 

In [ ]:
editrees_buff = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\env_data\Trees\Trees.shp")

In [ ]:
len(editrees_buff)

In [ ]:
editrees_buff["

#### setting tree values to mid range, setting wall abs to 0.1 

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

height_mapping = {
    "5 to 10 meters": 7.5,
    "10 to 15 meters": 12.5,
    "15 to 20 meters": 17.5,
    "20 to 25 meters": 22.5,
    "25 to 30 meters": 27.5,
    "Up to 5 meters": 4.5,
    "30 meters +": 32,
}

# Function to map height values
def extract_height(value):
    if pd.isna(value) or value is None:
        return None  # Keep NaN values

    value = str(value).strip()  # Convert to string and remove extra spaces
    
    if value in height_mapping:
        return height_mapping[value]  # Use predefined mapping
    
    return None  # If value is not found in mapping, return NaN
# Apply functions to extract values
editrees_buff["HEIGHT"] = editrees_buff["Height"].apply(extract_height)  # Tree height (meters)
editrees_buff["G"] = .1 # Wall absorption 

In [ ]:
editrees_buff["HEIGHT"].describe()

#### creating geometry buffer of 1 meter 
setting avg height for nan to 5 m 

In [ ]:
editrees_buff["geometry"] = editrees_buff.geometry.buffer(1)
editrees_buff['HEIGHT'] = editrees_buff['HEIGHT'].fillna(5)

#### combining trees and nm buildings 

In [ ]:
# Combine both GeoDataFrames
merged_gdf_buff = gpd.GeoDataFrame(pd.concat([editrees_buff, nm_edi_buildings], ignore_index=True))

# Ensure it remains a GeoDataFrame and retains geometries
merged_gdf_buff = gpd.GeoDataFrame(merged_gdf_buff, geometry="geometry", crs="EPSG:27700")

In [ ]:
print(len(nm_edi_buildings), len(editrees_buff), len(merged_gdf_buff))

In [ ]:
merged_gdf_buff.head()

#### subsetting to relevant fields 

In [ ]:
keep_col = ["HEIGHT", "geometry", "G"]
# Keep only the relevant columns
merged_gdf_buff = merged_gdf_buff[keep_col]

In [ ]:
merged_gdf_buff.head()

In [ ]:
merged_gdf_buff["PK"] = range(1, len(merged_gdf_buff)+1)

In [ ]:
merged_gdf_buff

In [ ]:
merged_gdf_buff.to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\noisemodelling_outputs\Round 6\buildings9_trees1_pk.geojson", driver="geoJSON")